# 📊 Dashboard de Portfólio — Bolsa Americana

Este notebook analisa seu portfólio de ações americanas com:
- 📦 Resumo geral (valor investido, valor atual, lucro/prejuízo)
- 📈 Evolução histórica do portfólio
- 🥧 Composição por ativo (pizza)
- 📊 Performance por ação (%)
- 🔥 Mapa de correlação entre ativos
- 📋 Tabela detalhada

---
**Instale as dependências antes de começar:**
```bash
pip install yfinance pandas plotly
```

## 📦 Célula 1 — Importações

In [41]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliotecas carregadas com sucesso!')

✅ Bibliotecas carregadas com sucesso!


## ⚙️ Célula 2 — Configuração do Portfólio

> **✏️ Edite aqui suas ações reais!**
> - `shares`: quantidade de ações que você possui
> - `avg_price`: seu preço médio de compra em dólares

In [42]:
# ============================================================
# 🔧 EDITE AQUI COM SEU PORTFÓLIO REAL
# ============================================================
PORTFOLIO = {
    "AAPL":  {"shares": 1.013, "avg_price": 198.03},
    "ASML":  {"shares": 0.3281,  "avg_price": 640.02},
    "GOOGL": {"shares": 0.7847,  "avg_price": 273.98},
    "EMBJ":  {"shares": 1,  "avg_price": 46.53},
    "AVGO":  {"shares": 0.3659, "avg_price": 210.00},
    "ENB":  {"shares": 3, "avg_price": 45.59},
    "SO":  {"shares": 2.1269, "avg_price": 94.82},
    "JNJ":  {"shares": 1, "avg_price": 157.94},
    "CVS":  {"shares": 3, "avg_price": 74.38},
    "KO":  {"shares": 2, "avg_price": 72.38},
    "PG":  {"shares": 1.4796, "avg_price": 150.07},
    "NTST":  {"shares": 4, "avg_price": 18.17},
    "O":  {"shares": 5, "avg_price": 57.57},
    "EQIX":  {"shares": 0.2975, "avg_price": 843.36},
    "PLD":  {"shares": 1, "avg_price": 99.16},
    "WPC":  {"shares": 2, "avg_price": 61.95},
    "NU":  {"shares": 16.7574, "avg_price": 14.69},
    "BDORY":  {"shares": 10, "avg_price": 4.70},
    "JPM":  {"shares": 0.3685, "avg_price": 312.00},
    "V":  {"shares": 0.8597, "avg_price": 325.65},
    "CSPX.L":  {"shares": 0.2821, "avg_price": 744.30},
    "VWRA.L":  {"shares": 1.094, "avg_price": 158.98},
    "XGDU.L":  {"shares": 4, "avg_price": 66.07},
}

# Período para histórico: '1mo', '3mo', '6mo', '1y', '2y', '5y'
PERIODO = "1y"
# ============================================================

TICKERS = list(PORTFOLIO.keys())
print(f"📋 Portfólio configurado com {len(TICKERS)} ativos: {', '.join(TICKERS)}")

📋 Portfólio configurado com 23 ativos: AAPL, ASML, GOOGL, EMBJ, AVGO, ENB, SO, JNJ, CVS, KO, PG, NTST, O, EQIX, PLD, WPC, NU, BDORY, JPM, V, CSPX.L, VWRA.L, XGDU.L


## 🌐 Célula 3 — Download dos Dados

Usa a biblioteca `yfinance` para buscar preços atuais e históricos direto do Yahoo Finance (gratuito).

In [43]:
def get_portfolio_data(portfolio: dict) -> pd.DataFrame:
    """Baixa preço atual, variação diária e calcula lucro/prejuízo de cada ativo."""
    rows = []
    print("⏳ Buscando dados do mercado...")

    for ticker, info in portfolio.items():
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="2d")

            if hist.empty:
                print(f"  ⚠️  Sem dados para {ticker}")
                continue

            current_price = hist["Close"].iloc[-1]
            prev_close    = hist["Close"].iloc[-2] if len(hist) > 1 else current_price
            day_chg_pct   = ((current_price - prev_close) / prev_close) * 100

            shares         = info["shares"]
            avg_price      = info["avg_price"]
            total_invested = shares * avg_price
            current_value  = shares * current_price
            profit_loss    = current_value - total_invested
            profit_pct     = ((current_value / total_invested) - 1) * 100

            rows.append({
                "Ticker":             ticker,
                "Preço Atual ($)":    round(current_price, 2),
                "Preço Médio ($)":    round(avg_price, 2),
                "Qtd. Ações":         shares,
                "Valor Investido ($)": round(total_invested, 2),
                "Valor Atual ($)":    round(current_value, 2),
                "Lucro/Prej. ($)":    round(profit_loss, 2),
                "Lucro/Prej. (%)": round(profit_pct, 2),
                "Var. Diária (%)": round(day_chg_pct, 2),
            })
            print(f"  ✅ {ticker}: ${current_price:.2f}  ({day_chg_pct:+.2f}% hoje)")

        except Exception as e:
            print(f"  ❌ Erro em {ticker}: {e}")

    return pd.DataFrame(rows)


def get_historical_prices(tickers: list, period: str) -> pd.DataFrame:
    """Baixa preços históricos ajustados para todos os tickers."""
    data = yf.download(tickers, period=period, auto_adjust=True, progress=False)["Close"]
    if isinstance(data, pd.Series):          # só 1 ticker
        data = data.to_frame(name=tickers[0])
    return data


# — Executa o download —
df = get_portfolio_data(PORTFOLIO)
prices = get_historical_prices(TICKERS, PERIODO)

print(f"\n📅 Histórico de {PERIODO}: {len(prices)} pregões carregados")

⏳ Buscando dados do mercado...
  ✅ AAPL: $255.12  (-2.18% hoje)
  ✅ ASML: $1352.18  (-2.49% hoje)
  ✅ GOOGL: $304.16  (-1.47% hoje)
  ✅ EMBJ: $57.77  (-11.25% hoje)
  ✅ AVGO: $336.64  (-1.44% hoje)
  ✅ ENB: $54.16  (+0.97% hoje)
  ✅ SO: $98.33  (+2.16% hoje)
  ✅ JNJ: $243.91  (+0.38% hoje)
  ✅ CVS: $76.41  (+0.91% hoje)
  ✅ KO: $77.99  (+0.46% hoje)
  ✅ PG: $151.25  (-1.35% hoje)
  ✅ NTST: $20.16  (-0.64% hoje)
  ✅ O: $65.12  (+0.63% hoje)
  ✅ EQIX: $978.03  (+0.47% hoje)
  ✅ PLD: $131.95  (-1.83% hoje)
  ✅ WPC: $72.12  (+0.90% hoje)
  ✅ NU: $14.03  (-3.14% hoje)
  ✅ BDORY: $4.71  (-5.23% hoje)
  ✅ JPM: $282.07  (-1.90% hoje)
  ✅ V: $306.55  (-0.78% hoje)
  ✅ CSPX.L: $719.97  (+0.00% hoje)
  ✅ VWRA.L: $169.36  (-1.11% hoje)
  ✅ XGDU.L: $78.94  (-0.69% hoje)

📅 Histórico de 1y: 258 pregões carregados


## 📋 Célula 4 — Resumo Geral do Portfólio

In [44]:
total_investido = df["Valor Investido ($)"].sum()
total_atual     = df["Valor Atual ($)"].sum()
lucro_total     = total_atual - total_investido
lucro_pct       = ((total_atual / total_investido) - 1) * 100
sinal           = "+" if lucro_total >= 0 else ""

print("=" * 45)
print("        📊 RESUMO DO PORTFÓLIO")
print("=" * 45)
print(f"  💰 Total Investido :  ${total_investido:>12,.2f}")
print(f"  📈 Valor Atual     :  ${total_atual:>12,.2f}")
print(f"  💹 Lucro/Prejuízo  :  {sinal}${lucro_total:>11,.2f}  ({sinal}{lucro_pct:.2f}%)")
print(f"  🎯 Nº de Ativos    :  {len(df)}")
print("=" * 45)

# Tabela completa
df.style \
  .map(lambda v: "color: green" if v > 0 else "color: red",
            subset=["Lucro/Prej. ($)", "Lucro/Prej. (%)", "Var. Diária (%)"]) \
  .format({
      "Preço Atual ($)":     "${:.2f}",
      "Preço Médio ($)":     "${:.2f}",
      "Valor Investido ($)": "${:,.2f}",
      "Valor Atual ($)":     "${:,.2f}",
      "Lucro/Prej. ($)":     "${:+,.2f}",
      "Lucro/Prej. (%)": "{:+.2f}%",
      "Var. Diária (%)": "{:+.2f}%",
  })

        📊 RESUMO DO PORTFÓLIO
  💰 Total Investido :  $    4,006.03
  📈 Valor Atual     :  $    4,673.54
  💹 Lucro/Prejuízo  :  +$     667.51  (+16.66%)
  🎯 Nº de Ativos    :  23


,Ticker,Preço Atual ($),Preço Médio ($),Qtd. Ações,Valor Investido ($),Valor Atual ($),Lucro/Prej. ($),Lucro/Prej. (%),Var. Diária (%)
0,AAPL,$255.12,$198.03,1.013000,$200.60,$258.44,$+57.83,+28.83%,-2.18%
1,ASML,$1352.18,$640.02,0.328100,$209.99,$443.65,$+233.66,+111.27%,-2.49%
2,GOOGL,$304.16,$273.98,0.784700,$214.99,$238.67,$+23.68,+11.02%,-1.47%
3,EMBJ,$57.77,$46.53,1.000000,$46.53,$57.77,$+11.24,+24.16%,-11.25%
4,AVGO,$336.64,$210.00,0.365900,$76.84,$123.18,$+46.34,+60.31%,-1.44%
5,ENB,$54.16,$45.59,3.000000,$136.77,$162.48,$+25.71,+18.80%,+0.97%
6,SO,$98.33,$94.82,2.126900,$201.67,$209.15,$+7.48,+3.71%,+2.16%
7,JNJ,$243.91,$157.94,1.000000,$157.94,$243.91,$+85.97,+54.43%,+0.38%
8,CVS,$76.41,$74.38,3.000000,$223.14,$229.23,$+6.09,+2.73%,+0.91%
9,KO,$77.99,$72.38,2.000000,$144.76,$155.98,$+11.22,+7.75%,+0.46%


## 📈 Célula 5 — Evolução Histórica do Portfólio

In [45]:
# Calcula o valor total do portfólio dia a dia
portfolio_value = pd.Series(0.0, index=prices.index)
for ticker, info in PORTFOLIO.items():
    if ticker in prices.columns:
        portfolio_value += prices[ticker] * info["shares"]

# Linha de referência: valor investido total
custo_base = sum(v["shares"] * v["avg_price"] for v in PORTFOLIO.values())

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=portfolio_value.index,
    y=portfolio_value.values,
    mode="lines",
    name="Valor do Portfólio",
    line=dict(color="#00d4aa", width=2.5),
    fill="tozeroy",
    fillcolor="rgba(0,212,170,0.12)",
))

fig.add_hline(
    y=custo_base,
    line_dash="dash",
    line_color="#ff4d6d",
    annotation_text=f" Custo Base: ${custo_base:,.0f}",
    annotation_position="bottom right",
)

fig.update_layout(
    title=f"Evolução do Portfólio — últimos {PERIODO}",
    xaxis_title="Data",
    yaxis_title="Valor ($)",
    yaxis_tickprefix="$",
    template="plotly_dark",
    hovermode="x unified",
    height=450,
)
fig.show()

## 🥧 Célula 6 — Composição do Portfólio

In [46]:
fig = px.pie(
    df,
    values="Valor Atual ($)",
    names="Ticker",
    title="Composição do Portfólio (por valor atual)",
    hole=0.45,
    color_discrete_sequence=px.colors.sequential.Teal,
)

fig.update_traces(textposition="outside", textinfo="percent+label")
fig.update_layout(template="plotly_dark", height=450)
fig.show()

## 📊 Célula 7 — Performance por Ação (%)

In [47]:
df_sorted = df.sort_values("Lucro/Prej. (%)")
cores = ["#00d4aa" if v >= 0 else "#ff4d6d" for v in df_sorted["Lucro/Prej. (%)"]]

fig = go.Figure(go.Bar(
    x=df_sorted["Ticker"],
    y=df_sorted["Lucro/Prej. (%)"],
    marker_color=cores,
    text=[f"{v:+.2f}%" for v in df_sorted["Lucro/Prej. (%)"]],
    textposition="outside",
))

fig.update_layout(
    title="Performance Total por Ação",
    xaxis_title="Ticker",
    yaxis_title="Retorno (%)",
    yaxis_ticksuffix="%",
    template="plotly_dark",
    height=400,
)
fig.show()

## 📉 Célula 8 — Preços Históricos Individuais

Compara a evolução normalizada de cada ativo (base 100 = início do período).

In [48]:
# Normaliza para base 100 — facilita comparação entre ações de preços diferentes
prices_norm = (prices / prices.iloc[0]) * 100

fig = go.Figure()
for ticker in TICKERS:
    if ticker in prices_norm.columns:
        fig.add_trace(go.Scatter(
            x=prices_norm.index,
            y=prices_norm[ticker],
            mode="lines",
            name=ticker,
        ))

fig.add_hline(y=100, line_dash="dot", line_color="gray",
              annotation_text=" Base 100")

fig.update_layout(
    title=f"Performance Normalizada (base 100) — {PERIODO}",
    xaxis_title="Data",
    yaxis_title="Valor Normalizado",
    template="plotly_dark",
    hovermode="x unified",
    height=450,
)
fig.show()

## 🔥 Célula 9 — Mapa de Correlação

Valores próximos de **+1** = movem juntos | Próximos de **-1** = movem ao contrário

> 💡 **Dica de diversificação:** idealmente, você quer correlações baixas entre seus ativos para reduzir o risco.

In [49]:
# Retornos diários
returns = prices.pct_change().dropna()
corr    = returns.corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    title=f"Correlação dos Retornos Diários — {PERIODO}",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    aspect="auto",
)

fig.update_layout(template="plotly_dark", height=450)
fig.show()

# Par com menor correlação (melhor diversificação)
import numpy as np
mascara = np.triu(np.ones(corr.shape), k=1).astype(bool)
stacked = corr.where(mascara).stack()
stacked.index.names = ["Ativo A", "Ativo B"]
corr_pairs = stacked.reset_index(name="Correlação")
menor = corr_pairs.loc[corr_pairs["Correlação"].idxmin()]
print(f"\n🎯 Menor correlação: {menor['Ativo A']} × {menor['Ativo B']} = {menor['Correlação']:.2f}")


🎯 Menor correlação: AVGO × PG = -0.22


## 💾 Célula 10 — Exportar Portfólio para CSV

Salva um snapshot do portfólio com data e hora.

In [ ]:
#timestamp = datetime.now().strftime("%Y%m%d_%H%M")
#nome_arquivo = f"portfolio_{timestamp}.csv"

#df.to_csv(nome_arquivo, index=False, sep=";", decimal=",")
#print(f"✅ Arquivo salvo: {nome_arquivo}")
#print(f"📂 Colunas exportadas: {', '.join(df.columns)}")

✅ Arquivo salvo: portfolio_20260312_1852.csv
📂 Colunas exportadas: Ticker, Preço Atual ($), Preço Médio ($), Qtd. Ações, Valor Investido ($), Valor Atual ($), Lucro/Prej. ($), Lucro/Prej. (%), Var. Diária (%)
